# 6. Comparison Checkpoint: Handwritten Loop vs ToyAIKit

This notebook compares the explicit agent loop from notebook 05 with ToyAIKit's framework-managed version.

## Comparison checkpoint

| Approach | What you write | What the framework manages |
| --- | --- | --- |
| Handwritten loop | schemas, history, dispatch, stopping | nothing beyond the model client |
| ToyAIKit | tools, prompts, callbacks | loop, display, history, cost helpers |

The goal is to identify which responsibilities move into the framework, not to treat the framework as magic.

## 1. Setup

Install the teaching dependency in the project environment before running the import cell:

```bash
uv add toyaikit
```

ToyAIKit is already declared in this project's dependency configuration. If your current environment cannot import it, synchronize the environment before continuing.

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
from ingestion import build_index, load_faq_data

In [9]:
load_dotenv()
openai_client = OpenAI()
documents = load_faq_data()
index = build_index(documents)
print(f"Loaded and indexed {len(documents)} documents")

Loaded and indexed 1401 documents


In [10]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

## 2. Define the search tool

Start with the same search behavior used in notebook 04. The framework can register this function with the hand-written schema first, so you can compare explicit and generated tool descriptions.

In [11]:
def search(query):
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

## 3. Register the tool with its hand-written schema

`Tools.add_tool` connects the Python callable with the schema the model sees. This is the framework equivalent of passing `tools=[search_tool]` to `responses.create(...)`.

In [12]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

## 4. Let ToyAIKit generate the schema

Type hints describe the argument shape and the docstring describes the tool's purpose. ToyAIKit can derive the JSON schema instead of requiring a hand-written dictionary.

In [13]:
def typed_search(query: str) -> list[dict]:
    """Search the FAQ database for matching course information."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )


generated_tools = Tools()
generated_tools.add_tool(typed_search)
generated_tools.get_tools()

[{'type': 'function',
  'name': 'typed_search',
  'description': 'Search the FAQ database for matching course information.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## 5. Create the chat interface and runner

The chat interface renders messages and tool calls in the notebook. The runner owns the same loop you implemented by hand: send messages, execute tools, append results, and repeat until the model returns a final answer.

In [14]:
instructions = """
You're a course teaching assistant.
Use the search tool for course questions.
Search again with better keywords when the first result is incomplete.
Answer only from the FAQ context.
""".strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=generated_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

## 6. Run the typo-recovery prompt

Use `Olama` deliberately. Watch the callback output: the useful evidence is the sequence of tool calls and messages, not only the final answer.

In [15]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
result

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nUse the search tool for course questions.\nSearch again with better keywords when the first result is incomplete.\nAnswer only from the FAQ context.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama run locally FAQ"}', call_id='call_4BDHWuYVqlGxrcxwel7XwUfb', name='typed_search', type='function_call', id='fc_03be559ca1b77346006a74a93ccbd0819e8c9e6a4fac1a01e5', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_4BDHWuYVqlGxrcxwel7XwUfb', 'output': '[\n  {\n    "id": "aa310de435",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Can I run the course locally instead of Codespaces?",\n    "answer": "Yes. Codespaces is just the easiest way for everyone to start with the same environment.\\n\\nYou can

## 7. Inspect cost and message history

ToyAIKit exposes the same operational information that the handwritten loop required you to collect manually.

In [16]:
print("Cost:", result.cost)
print("Message count:", len(result.all_messages))
result.cost, result.all_messages

Cost: CostInfo(input_cost=Decimal('0.00102825'), output_cost=Decimal('0.0006615'), total_cost=Decimal('0.00168975'))
Message count: 5


(CostInfo(input_cost=Decimal('0.00102825'), output_cost=Decimal('0.0006615'), total_cost=Decimal('0.00168975')),
 [EasyInputMessage(content="You're a course teaching assistant.\nUse the search tool for course questions.\nSearch again with better keywords when the first result is incomplete.\nAnswer only from the FAQ context.", role='developer', phase=None, type=None),
  EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
  ResponseFunctionToolCall(arguments='{"query":"Olama run locally FAQ"}', call_id='call_4BDHWuYVqlGxrcxwel7XwUfb', name='typed_search', type='function_call', id='fc_03be559ca1b77346006a74a93ccbd0819e8c9e6a4fac1a01e5', namespace=None, status='completed'),
  {'type': 'function_call_output',
   'call_id': 'call_4BDHWuYVqlGxrcxwel7XwUfb',
   'output': '[\n  {\n    "id": "aa310de435",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Can I run the course locally instead of Codespaces?",\n    "answer": 

## 8. Continue the conversation

Pass `result.all_messages` as `previous_messages` so the runner can resolve the follow-up reference to Ollama.

In [17]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
result2

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content='How do I run a different model?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run a different model FAQ llm-zoomcamp model provider swap OpenAI Gemini Anthropic"}', call_id='call_MAild3m9jr0yNLlWDofVyPt2', name='typed_search', type='function_call', id='fc_03be559ca1b77346006a74a974c2fc819e8efb776103d3c1ca', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_MAild3m9jr0yNLlWDofVyPt2', 'output': '[\n  {\n    "id": "ee43413718",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Can I use a model or provider different from the one recommended in homework?",\n    "answer": "Yes. The recommended model is not mandatory. You can use OpenAI, Gemini, Groq, OpenRouter, Azure OpenAI, local models, or another provider.\\n\\nThe homework is designed so you do not need a paid service. You may need to adapt the code for your provider, because res

## 9. Optional interactive chat

This starts ToyAIKit's notebook input loop. Run it only when you are ready to enter questions interactively; type `stop` to exit.

In [18]:
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nUse the search tool for course questions.\nSearch again with better keywords when the first result is incomplete.\nAnswer only from the FAQ context.", role='developer', phase=None, type=None), EasyInputMessage(content='what is the cost of the course', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"course cost tuition fee price"}', call_id='call_7REk0Y6jTsTyP6t70vFWjY7d', name='typed_search', type='function_call', id='fc_0242b2f904ecdfcc006a74a99174bc81a38c973ccb50ea5014', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_7REk0Y6jTsTyP6t70vFWjY7d', 'output': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n   

## 10. Study checklist

Compare this notebook with `05-agentic-loop.ipynb`:

- Which lines disappeared because ToyAIKit owns the loop?
- Where is the tool schema created?
- Where can you inspect cost and history?
- What happens if the callback or tool execution fails?
- Why is ToyAIKit useful for learning but not automatically a production choice?

Frameworks package the loop; they do not remove the need to understand tools, prompts, history, stopping conditions, cost, and safety.